In [1]:
!pip install datasets==2.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 9.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
from datasets import load_dataset

In [3]:
def load_hellaswag_lora():
    dataset = load_dataset("hellaswag", split="train")

    data = []

    for item in tqdm(dataset):
        options = item["endings"]
        label = int(item["label"])

        options_text = "\n".join(
            f"{i}. {opt}" for i, opt in enumerate(options)
        )

        user_prompt = f"""Choose the correct option. Answer with ONLY the digit (0, 1, 2, or 3).

Question:
{item['ctx']}

Options:
{options_text}
"""

        assistant = str(label)

        data.append({
            "messages": [
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant}
            ]
        })

    return data

In [4]:
def correct_answer_format_mmlu(ans, options):
    idx = int(ans)
    letter = ["A", "B", "C", "D"][idx]
    return f"Answer: {letter}. {options[idx]}"


def load_mmlu_lora():
    dataset = load_dataset("cais/mmlu", "all", split="test")

    data = []

    for item in tqdm(dataset):
        options = item["choices"]
        answer = correct_answer_format_mmlu(item["answer"], options)

        options_text = "\n".join(
            f"{l}. {opt}" for l, opt in zip(["A","B","C","D"], options)
        )
        #options = f"A. {row['options'][0]}\nB. {row['options'][1]}\nC. {row['options'][2]}\nD. {row['options'][3]}\n"

        user_prompt = f"""Answer the question and choose one of the suggested answers. 
Answer format: "Answer: <letter>. Text of the selected option".

Question:
{item['question']}

Options:
{options_text}
"""

        data.append({
            "messages": [
                {"role": "system", "content": "You are a knowledgeable assistant."},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": answer}
            ]
        })

    return data

In [5]:
def load_dailymail_lora():
    dataset = load_dataset("abisee/cnn_dailymail", "2.0.0", split="train")

    data = []

    for item in tqdm(dataset):
        article = item["article"]

        summary = " ".join(item["highlights"])

        user_prompt = f"""Summarize the following news article.

Article:
{article}

Write a concise summary in 3-4 sentences.
Ensure the summary is factually accurate and based only on the article.
Do not add any external information or assumptions.
"""

        data.append({
            "messages": [
                {"role": "system", "content": "You summarize texts accurately."},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": summary}
            ]
        })

    return data

In [6]:
def load_hover_lora():
    dataset = load_dataset("hover")
    dataset = dataset["validation"]

    data = []
    
    for item in tqdm(dataset):
        claim = item["claim"]
        label = item["label"]

        if label == 0:
            answer = "SUPPORTS"
        elif label == 1:
            answer = "REFUTES"
        else:
            answer = "NOT ENOUGH INFORMATION"

        user_prompt = f"""Determine if the claim is true.

Claim:
{claim}

Answer with:
SUPPORTS or REFUTES or NOT ENOUGH INFORMATION
"""

        data.append({
            "messages": [
                {"role": "system", "content": "You are a fact-checking assistant."},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": answer}
            ]
        })

    return data

формирование датасета для дообучения

In [7]:
def build_lora_dataset(output_path="lora_dataset.jsonl", max_samples_per_dataset=None):
    all_data = []

    datasets = [
        #load_hover_lora(),
        load_hellaswag_lora(),
        load_mmlu_lora(),
        load_dailymail_lora()
        
    ]

    for ds in datasets:
        if max_samples_per_dataset:
            ds = ds[:max_samples_per_dataset]
        all_data.extend(ds)

    print(f"Total samples: {len(all_data)}")

    with open(output_path, "w", encoding="utf-8") as f:
        for item in all_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Path: {output_path}")

In [8]:
build_lora_dataset(
    output_path="lora_dataset.json",
    max_samples_per_dataset=15000   
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39905 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

100%|██████████| 287113/287113 [00:13<00:00, 21207.37it/s]


Total samples: 44042
Path: lora_dataset.json
